# Talaffuz.ai on Google Colab

<a href="https://colab.research.google.com/github/XOVO-Technologies/talaffuz.ai/blob/main/notebooks/talaffuz_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the app on a Colab machine and opens it in your browser. You drop in long Urdu recordings. It cuts them into sentence-length clips, removes noise and music, transcribes each clip, and exports numbered wav files with `metadata.csv` and `metadata.xlsx`.

**Before you start**

1. Open the menu **Runtime > Change runtime type** and pick a GPU (a T4 is enough). Without one the notebook still works, but each clip takes about 35 seconds instead of a few.
2. Run the cells from top to bottom, or use **Runtime > Run all**. Every cell says what it does and what to do next.
3. Keep this tab open while the app is processing. Colab stops idle sessions.

Setup takes several minutes the first time, because it installs the Python packages and builds the web page.

## 1. Check the machine

In [ ]:
#@title 1. Check the machine { display-mode: "form" }
import shutil
import subprocess

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""

if gpu:
    print("GPU found:", gpu)
else:
    print("No GPU found. The app will run on the CPU, at about 35 seconds per clip.")
    print("For speed: Runtime > Change runtime type > pick a GPU, then run this cell again.")
print(f"Free disk space: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

## 2. Install the app

This downloads the code from GitHub, installs the Python packages and builds the web page. It is safe to run again after a disconnect.

If you use a fork, change `REPO` to your own `owner/name`.

In [ ]:
#@title 2. Install the app { display-mode: "form" }
import os
import subprocess

REPO = "XOVO-Technologies/talaffuz.ai"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
APP_DIR = "/content/talaffuz.ai"


def run(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")


def node_ok():
    try:
        out = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip().lstrip("v")
        major, minor = (int(x) for x in out.split(".")[:2])
    except Exception:
        return False
    return (major == 20 and minor >= 19) or major >= 22


if os.path.isdir(APP_DIR):
    run(f"git -C {APP_DIR} pull --ff-only")
else:
    run(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git {APP_DIR}")

run(f"pip install -q -r {APP_DIR}/backend/requirements.txt")

if not node_ok():
    print("Installing a newer Node.js to build the web page ...")
    run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null")

run("npm ci --no-audit --no-fund --loglevel=error && npm run build", cwd=f"{APP_DIR}/frontend")
print("Install finished.")

## 3. Choose where projects are saved

A Colab machine is wiped when the session ends. If you save projects to Google Drive, you can come back later, run the cells again and press **Continue** in the app. Colab will ask for permission to use your Drive.

Drive space is used by your recordings, a working copy of each one, and the clips. Plan for a few times the size of your recordings. Untick the box to keep everything on the Colab machine instead, and download the dataset before the session ends.

In [ ]:
#@title 3. Where to save projects { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "talaffuz.ai"  #@param {type:"string"}

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}/data"
else:
    DATA_DIR = "/content/data"

os.makedirs(DATA_DIR, exist_ok=True)
print("Projects are saved in:", DATA_DIR)

## 4. Roman Urdu key (optional)

Claude Code does not run inside Colab, so Roman Urdu comes from an Anthropic API key if you have one. You have two ways to add it:

- Paste the key later in the app's **Review** step, where you can also pick the model. The app shows the prices and an estimate, and the lowest-cost model is the default.
- Or add it here as a secret: open the key icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY`, paste the key, and switch on **Notebook access**. The key is never printed or saved in the notebook.

No key is fine too. You can type Roman Urdu in the review table, or finish it later on your own computer: copy the project folder from Drive into the `data/projects/` folder of a local install and run `/romanize` in Claude Code.

In [ ]:
#@title 4. Roman Urdu key (optional) { display-mode: "form" }
try:
    from google.colab import userdata

    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key found. The app will write Roman Urdu after transcription.")
except Exception:
    print("No ANTHROPIC_API_KEY secret found. That is fine. Type Roman Urdu in the app, or finish it later with /romanize.")

## 5. Start the app

The first start downloads the Whisper model, about 1.6 GB, so give it a few minutes. When it is ready this cell prints a link. Open the link in a new tab, which is roomier than the embedded view.

In [ ]:
#@title 5. Start the app { display-mode: "form" }
import sys
import time
import urllib.request

PORT = 8000


def healthy():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3).status == 200
    except Exception:
        return False


if not healthy():
    env = dict(os.environ, DATASET_DATA_DIR=DATA_DIR, PYTHONUTF8="1", HF_HUB_DISABLE_SYMLINKS_WARNING="1")
    log = open("/content/server.log", "w")
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--port", str(PORT)],
                              cwd=f"{APP_DIR}/backend", env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        if healthy():
            break
        time.sleep(1)
    else:
        print(open("/content/server.log").read()[-3000:])
        raise RuntimeError("The app did not start. The log above shows why.")

from google.colab import output

print("The app is running. Open it here:")
output.serve_kernel_port_as_window(PORT)

## 6. Use the app

1. **Add recordings.** Drag audio files onto the page. For big files, open **Add files that are already on this computer** and paste a Drive path such as `/content/drive/MyDrive/recordings`. The files are copied into the project, so nothing is uploaded through the browser. If you turned Drive off in step 3, upload files in the Colab file browser (left sidebar) and paste their path instead.
2. **Start.** Press Start in step 3 of the app. Progress is saved after every clip. You can press Stop and Continue.
3. **Review.** Play each clip, fix the Urdu, and untick Keep on bad clips.
4. **Build dataset.** Press Build dataset in step 5 of the app, then run the cell below to download it.

If the session disconnects, run steps 2 to 5 again and press Continue in the app. Everything finished before the disconnect is still there as long as you saved to Drive.

Prefer to stay in the notebook? The next cell shows the app inline.

In [ ]:
#@title Optional: show the app inside this notebook { display-mode: "form" }
from google.colab import output

output.serve_kernel_port_as_iframe(PORT, height=950)

## 7. Get your dataset

Run this cell after pressing **Build dataset** in the app. It downloads `dataset.zip`, which holds the `dataset/` folder (the wavs and `metadata.csv`) and `metadata.xlsx`. If the browser download fails for a large file, copy it from your Drive folder instead. The path is printed below.

In [ ]:
#@title 7. Download the dataset { display-mode: "form" }
import glob

zips = sorted(glob.glob(f"{DATA_DIR}/projects/*/output/dataset.zip"), key=os.path.getmtime)
if not zips:
    print("No export found yet. In the app, press Build dataset in step 5, then run this cell again.")
else:
    latest = zips[-1]
    print(f"Latest export: {latest} ({os.path.getsize(latest) / 1e6:.0f} MB)")
    from google.colab import files

    files.download(latest)

## When you are done

Run the cell below to stop the app. Then use **Runtime > Disconnect and delete runtime** so the GPU is free for other people.

In [ ]:
#@title Stop the app { display-mode: "form" }
try:
    server.terminate()
    print("App stopped.")
except NameError:
    print("The app was not started from this notebook session.")